In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
import pickle
import os

print("Loading dataset...")
df = pd.read_csv('image_metadata.csv')

# Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')

# Features for modeling
print("Preprocessing data...")
# Extract time-based features
df['hour'] = df['timestamp'].dt.hour
df['dayofweek'] = df['timestamp'].dt.dayofweek

# One-hot encode location_id
df = pd.get_dummies(df, columns=['location_id'], drop_first=False)
features = ['person_count', 'hour', 'dayofweek'] + [col for col in df.columns if 'location_id_' in col]

scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

# Save scaler for later use in app.py
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    
# Create sequences for LSTM
SEQUENCE_LENGTH = 10

def create_sequences(data, labels, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length + 1):
        X.append(data[i:(i + seq_length)])
        y.append(labels[i + seq_length - 1])
    return np.array(X), np.array(y)

X, y = create_sequences(df[features].values, df['anomaly_label'].values, SEQUENCE_LENGTH)

# Split into train and test
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

# Calculate class weights to handle imbalance
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print(f"Class Weights: {class_weights}")

# Define LSTM model
model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, X_train.shape[2])),
    LSTM(64, activation='relu', return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Using a smaller learning rate can help stable training
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

print("Training model...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

print("Evaluating model...")
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

# Save the model
model.save('lstm_model.keras')
print("Model saved to lstm_model.keras")

# Save training history to show in Streamlit (Diagnostics)
with open('training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)
print("Training history saved.")
